### 1. Extract Folder Topic Names

In [50]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re

In [51]:
dataset_dir = Path('./20_newsgroups')
topic_names = sorted([topic.name for topic in dataset_dir.iterdir() if topic.is_dir()])
print(f"Number of topics: {len(topic_names)}")

Number of topics: 20


### 2. Data Loading Pipeline

In [52]:
def clean_newsgroup_body(raw_text):
    parts = raw_text.split('\n\n', 1) # headers from body
    body = parts[1] if len(parts) > 1 else raw_text
    text = body.lower()
    text = re.sub(r'^\s*[|>#]{2,}\s*$', '', text, flags=re.MULTILINE) # remove lines starting with >, |, # (common in quoted text)
    text = re.sub(r'(^|\n)\s*>[^\n]*', '', text) # remove quoted text, > & spaces
    text = re.sub(r'[^a-z\s]', ' ', text) # remove non-alphabetic characters, keep spaces
    text = re.sub(r'\s+', ' ', text).strip() # remove extra spaces and newlines, replace with single space
    text = re.sub(r'\S*@\S*\s?', '', text) # remove email addresses
    text = re.sub(r'\s+', ' ', text).strip() # removes white spaces and newlines, replaces with single space
    return text

In [53]:
def load_dataset_to_dataframe(folder_path):
    data_records = []
    
    for label_dir in os.listdir(folder_path): # iterate through root newsgroup folders
        dir_path = os.path.join(folder_path, label_dir)
        
        if not os.path.isdir(dir_path): # skipping non-directory files if any
            continue
            
        for file_name in os.listdir(dir_path): # iterate through files in each category folder
            file_path = os.path.join(dir_path, file_name)
            try:
                with open(file_path, "r", encoding="latin-1") as f: # read and use latin-1 encoding to prevent UnicodeDecodeErrors
                    raw_content = f.read()
                    cleaned_text = clean_newsgroup_body(raw_content) # clean the text using the defined function
                    if cleaned_text:
                        data_records.append({"file_id": file_name, "label": label_dir, 
                        "text": cleaned_text, "original_length": len(raw_content)}) # append only cleaned text
            except Exception as e:
                print(f"Failed to read {file_path}: {e}")
                
    return pd.DataFrame(data_records) # convert dict list of records to a DataFrame

In [54]:
dataset_path = "./20_newsgroups" 
df = load_dataset_to_dataframe(dataset_path)

print(f"Successfully loaded {len(df)} documents.")
print("\nClass Distribution:")
print(df['label'].value_counts())

Successfully loaded 19955 documents.

Class Distribution:
label
talk.politics.mideast       1000
talk.politics.misc          1000
talk.politics.guns          1000
talk.religion.misc          1000
alt.atheism                  999
rec.sport.baseball           999
rec.sport.hockey             999
sci.crypt                    999
sci.electronics              999
sci.space                    999
rec.autos                    998
sci.med                      998
comp.windows.x               998
rec.motorcycles              997
comp.graphics                997
comp.sys.ibm.pc.hardware     997
soc.religion.christian       997
comp.sys.mac.hardware        996
comp.os.ms-windows.misc      992
misc.forsale                 991
Name: count, dtype: int64


In [56]:
df.shape

(19955, 4)

In [57]:
df.head()

,file_id,label,text,original_length
0,75895,talk.politics.mideast,in article apr ncsu edu hernlem chess ncsu edu...,1744
1,76248,talk.politics.mideast,ab z virginia edu andi beyer writes uh oh the ...,1602
2,76277,talk.politics.mideast,andrew varvel writes ah c mon give the guy thr...,1810
3,76045,talk.politics.mideast,in article apr thunder mcrcim mcgill edu hasan...,2544
4,77197,talk.politics.mideast,srinivas suder writes precisely why not cuba w...,2365


In [69]:
random_texts = df['text'].sample(n=10)

for i, text in enumerate(random_texts, 1):
    print(f"========== RANDOM DOCUMENT {i} ==========")
    print(text + "...\n")

========== RANDOM DOCUMENT 1 ==========
once upon a time long long ago in this news group someone posted a schematic for a bit a d converter well i just found a use for the little monster anyone out there still got this text file it had a flip flop a resistor and a cap and a comparator op amp i think i would be extremely thankful to anyone who could mail me the schematic or post it to the news group o o beware the light at the end of the tunnel it may be an oncoming dragon m d c u mcorbin oucsace cs ohiou edu...

========== RANDOM DOCUMENT 2 ==========
here is a press release from the reserve officers association reserve officers say demographics ignored in nominations to close naval marine reserve centers to national desk defense writer contact herbert m hart of the reserve officers association of the united states washington april u s newswire the reserve officers association of the united states has alerted the defense base realignment and closure commission that the services failed